# Hacer resumenes

## Librerias

In [16]:
import torch
from pathlib import Path
import pandas as pd
import numpy as np
import json
import csv

from datasets import Dataset
import evaluate

from transformers import (
    BartForConditionalGeneration,
    BartTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


Torch: 2.5.1+cpu
CUDA available: False


## Configuración

In [ ]:
CFG = {
    'model_name': "facebook/bart-base",
    'output_dir': "2_Models/summarization",
    'max_input_length': 1024,
    'max_target_length': 128,
    'num_epochs': 3,
    'learning_rate': 2e-5,
    'batch_size': 4,
    'gradient_accumulation_steps': 4,
    'warmup_steps': 500,
    'weight_decay': 0.01,
    'save_steps': 500,
    'eval_steps': 500,
    'logging_steps': 100,
}
Path(CFG['output_dir']).mkdir(parents=True, exist_ok=True)

GEN_CFG = {
    "max_new_tokens": 60,
    "min_new_tokens": 12,
    "num_beams": 4,
    "early_stopping": True,
    "no_repeat_ngram_size": 3,
    "encoder_no_repeat_ngram_size": 3,
    "repetition_penalty": 1.2,
    "length_penalty": 0.9,
    "renormalize_logits": True,
}

print(CFG)
print(GEN_CFG)


{'model_name': 'facebook/bart-base', 'output_dir': '2_Models/summarization', 'max_input_length': 1024, 'max_target_length': 128, 'num_epochs': 3, 'learning_rate': 2e-05, 'batch_size': 4, 'gradient_accumulation_steps': 4, 'warmup_steps': 500, 'weight_decay': 0.01, 'save_steps': 500, 'eval_steps': 500, 'logging_steps': 100}
{'max_new_tokens': 60, 'min_new_tokens': 12, 'num_beams': 4, 'early_stopping': True, 'no_repeat_ngram_size': 3, 'encoder_no_repeat_ngram_size': 3, 'repetition_penalty': 1.2, 'length_penalty': 0.9, 'renormalize_logits': True}


## Modelo y token

In [4]:
FT_DIR = Path(CFG['output_dir']) / "final_model"
tokenizer = BartTokenizer.from_pretrained(FT_DIR)
model_infer = BartForConditionalGeneration.from_pretrained(FT_DIR)  
model_infer.to(DEVICE)


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

In [5]:
def setup_model_and_tokenizer(cfg=CFG):
    tok = BartTokenizer.from_pretrained(cfg['model_name'])
    model = BartForConditionalGeneration.from_pretrained(cfg['model_name'])
    model.to(DEVICE)
    rouge = evaluate.load("rouge")
    return model, tok, rouge

model, tokenizer, rouge = setup_model_and_tokenizer(CFG)


## Carga

In [6]:
def load_splits(csv_path: str, seed: int = 42):
    df = pd.read_csv(csv_path).sample(frac=1, random_state=seed).reset_index(drop=True)
    n = len(df)
    n_train = int(0.8*n)
    n_val   = int(0.1*n)
    train_df = df.iloc[:n_train]
    val_df   = df.iloc[n_train:n_train+n_val]
    test_df  = df.iloc[n_train+n_val:]
    print(f"Train={len(train_df)}  Val={len(val_df)}  Test={len(test_df)}")
    return train_df, val_df, test_df


## Preprocesamiento y métricas

In [7]:
def preprocess_batch(examples, tokenizer, cfg=CFG):
    X = tokenizer(
        examples['input_text'],
        max_length=cfg['max_input_length'],
        truncation=True,
        padding='max_length'
    )
    Y = tokenizer(
        examples['target_summary'],
        max_length=cfg['max_target_length'],
        truncation=True,
        padding='max_length'
    )
    X['labels'] = Y['input_ids']
    return X

In [8]:
def build_metrics_fn(tokenizer, rouge):
    def _metrics(eval_pred):
        preds, labels = eval_pred
        decoded_preds = []
        for p in preds:
            p_clean = [t for t in p if t is not None and t != -100]
            decoded_preds.append(tokenizer.decode(p_clean, skip_special_tokens=True))
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        res = rouge.compute(
            predictions=decoded_preds,
            references=decoded_labels,
            use_stemmer=True
        )
        return {'rouge1': res['rouge1'], 'rouge2': res['rouge2'], 'rougeL': res['rougeL']}
    return _metrics


In [9]:
def make_training_args(cfg):
    from transformers import Seq2SeqTrainingArguments
    import torch
    try:
        return Seq2SeqTrainingArguments(
            output_dir=str(cfg['output_dir']),
            evaluation_strategy="steps",
            eval_steps=cfg['eval_steps'],
            save_strategy="steps",
            save_steps=cfg['save_steps'],
            learning_rate=cfg['learning_rate'],
            per_device_train_batch_size=cfg['batch_size'],
            per_device_eval_batch_size=cfg['batch_size'],
            gradient_accumulation_steps=cfg['gradient_accumulation_steps'],
            num_train_epochs=cfg['num_epochs'],
            warmup_steps=cfg['warmup_steps'],
            weight_decay=cfg['weight_decay'],
            logging_steps=cfg['logging_steps'],
            predict_with_generate=True,
            generation_max_length=cfg['max_target_length'],
            load_best_model_at_end=True,
            metric_for_best_model="rougeL",
            greater_is_better=True,
            save_total_limit=2,
            fp16=False,  
        )
    except TypeError:
        try:
            return Seq2SeqTrainingArguments(
                output_dir=str(cfg['output_dir']),
                eval_strategy="steps",
                eval_steps=cfg['eval_steps'],
                save_steps=cfg['save_steps'],
                learning_rate=cfg['learning_rate'],
                per_device_train_batch_size=cfg['batch_size'],
                per_device_eval_batch_size=cfg['batch_size'],
                gradient_accumulation_steps=cfg['gradient_accumulation_steps'],
                num_train_epochs=cfg['num_epochs'],
                warmup_steps=cfg['warmup_steps'],
                weight_decay=cfg['weight_decay'],
                logging_steps=cfg['logging_steps'],
                predict_with_generate=True,
                generation_max_length=cfg['max_target_length'],
                load_best_model_at_end=True,
                metric_for_best_model="rougeL",
                greater_is_better=True,
                save_total_limit=2,
                fp16=False,
            )
        except TypeError:
            return Seq2SeqTrainingArguments(
                output_dir=str(cfg['output_dir']),
                eval_strategy="steps",
                eval_steps=cfg['eval_steps'],
                save_steps=cfg['save_steps'],
                learning_rate=cfg['learning_rate'],
                per_device_train_batch_size=cfg['batch_size'],
                per_device_eval_batch_size=cfg['batch_size'],
                gradient_accumulation_steps=cfg['gradient_accumulation_steps'],
                num_train_epochs=cfg['num_epochs'],
                warmup_steps=cfg['warmup_steps'],
                weight_decay=cfg['weight_decay'],
                logging_steps=cfg['logging_steps'],
                predict_with_generate=True,
                generation_max_length=cfg['max_target_length'],
                load_best_model_at_end=True,
                save_total_limit=2,
                fp16=False,
            )


## Entrenamiento

In [10]:
def train_model(model, tokenizer, train_df, val_df, cfg=CFG):
    from datasets import Dataset
    from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq
    import json
    from pathlib import Path

    print("="*60, "\nINICIANDO FINE-TUNING\n", "="*60, sep='')
    train_ds = Dataset.from_pandas(train_df[['input_text', 'target_summary']])
    val_ds   = Dataset.from_pandas(val_df[['input_text', 'target_summary']])

    train_ds = train_ds.map(
        lambda b: preprocess_batch(b, tokenizer, cfg),
        batched=True,
        remove_columns=['input_text', 'target_summary']
    )
    val_ds = val_ds.map(
        lambda b: preprocess_batch(b, tokenizer, cfg),
        batched=True,
        remove_columns=['input_text', 'target_summary']
    )

    args = make_training_args(cfg)
    collator = DataCollatorForSeq2Seq(tokenizer, model=model)
    metrics_fn = build_metrics_fn(tokenizer, rouge)  

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=metrics_fn,
    )

    train_result = trainer.train()

    out_dir = Path(cfg['output_dir']) / "final_model"
    out_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(out_dir))
    tokenizer.save_pretrained(str(out_dir))

    with open(Path(cfg['output_dir']) / "training_metrics.json", "w") as f:
        json.dump(train_result.metrics, f, indent=2)

    print("\n✓ Fine-tuning completado.")
    return trainer


## Evaluación

In [11]:
def evaluate_model(trainer, tokenizer, test_df, cfg=CFG):
    print("="*60, "\nEVALUACIÓN EN TEST\n", "="*60, sep='')
    test_ds = Dataset.from_pandas(test_df[['input_text', 'target_summary']])
    test_ds = test_ds.map(
        lambda b: preprocess_batch(b, tokenizer, cfg),
        batched=True,
        remove_columns=['input_text', 'target_summary']
    )
    metrics = trainer.evaluate(test_ds)
    for k, v in metrics.items():
        try:
            print(f"{k}: {float(v):.4f}")
        except Exception:
            print(f"{k}: {v}")
    with open(Path(cfg['output_dir']) / "test_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics


## Generador de ejemplos

In [12]:
@torch.inference_mode()
def generate_samples_table(model, tokenizer, df, n_samples=5, cfg=CFG, device=DEVICE):
    import pandas as pd
    rows, k = [], min(n_samples, len(df))
    for i in range(k):
        src = str(df.iloc[i]['input_text'])
        tgt = str(df.iloc[i]['target_summary'])
        enc = tokenizer(src, max_length=cfg.get('max_input_length', 512), truncation=True, return_tensors="pt").to(device)

        out = model.generate(
            **enc,
            max_new_tokens=GEN_CFG["max_new_tokens"],
            min_new_tokens=GEN_CFG["min_new_tokens"],
            num_beams=GEN_CFG["num_beams"],
            early_stopping=GEN_CFG["early_stopping"],
            no_repeat_ngram_size=GEN_CFG["no_repeat_ngram_size"],
            encoder_no_repeat_ngram_size=GEN_CFG["encoder_no_repeat_ngram_size"],
            repetition_penalty=GEN_CFG["repetition_penalty"],
            length_penalty=GEN_CFG["length_penalty"],
            renormalize_logits=GEN_CFG["renormalize_logits"],
        )
        gen = tokenizer.decode(out[0], skip_special_tokens=True)

        rows.append({
            "input_full": src,
            "input_preview": (src[:300] + "...") if len(src) > 300 else src,
            "target": tgt,
            "generated": gen,
            "len_input": len(src),
            "len_target": len(tgt),
            "len_generated": len(gen),
        })
    return pd.DataFrame(rows)


## Muestra de uso

In [13]:
#Cargar splits
train_df, val_df, test_df = load_splits("C:/Users/Alina Tatjana/OneDrive/Documentos/UVG/8vo SEMESTRE/Deep Learning/Proyecto/Proyecto/1_Data/processed/summarization_data.csv")

Train=530  Val=66  Test=67


In [ ]:
# Entrenar
trainer = train_model(model, tokenizer, train_df, val_df)


INICIANDO FINE-TUNING


Map: 100%|██████████| 39/39 [00:00<00:00, 84.11 examples/s]


Step,Training Loss,Validation Loss


c:\Users\Alina Tatjana\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(



✓ Fine-tuning completado.


In [ ]:
metrics = evaluate_model(trainer, tokenizer, test_df)

EVALUACIÓN EN TEST


Map: 100%|██████████| 39/39 [00:00<00:00, 95.69 examples/s]


eval_loss: 7.6291
eval_rouge1: 0.4079
eval_rouge2: 0.2902
eval_rougeL: 0.3354
eval_runtime: 242.8639
eval_samples_per_second: 0.1610
eval_steps_per_second: 0.0410
epoch: 3.0000


In [14]:
df_samples = generate_samples_table(model_infer, tokenizer, test_df, n_samples=3)
df_samples[["input_preview", "target", "generated", "len_input", "len_target", "len_generated"]]


,input_preview,target,generated,len_input,len_target,len_generated
0,Los Angeles Lakers forward LeBron James 23 pre...,LeBron James injury information leaked to bett...,Los Los Angeles Clippers forward LeBronJames 2...,1687,238,289
1,"At present, Hims Hers Health generates 159 mil...",Is Hims Hers Health Still a Smart Opportunity ...,"At Present, Hes Hers Holdings is trading on a ...",8630,236,233
2,Jamaican officials issued dire warnings Saturd...,Hurricane Melissa takes aim at Jamaica. Jamaic...,JAMAican officials warned dire warnings issued...,243,225,262


## Guardar en nuevo CSV

In [ ]:
df_samples = generate_samples_table(model_infer, tokenizer, test_df, n_samples=len(test_df))

orig_path = r"C:/Users/Alina Tatjana/OneDrive/Documentos/UVG/8vo SEMESTRE/Deep Learning/Proyecto/Proyecto/1_Data/processed/summarization_data.csv"
df_orig = pd.read_csv(orig_path)


df_out = df_orig.copy()
k = len(df_samples)

if "generated" not in df_out.columns:
    df_out["generated"] = ""
if "target_preview" not in df_out.columns:
    df_out["target_preview"] = ""

df_out.loc[:k-1, "generated"] = df_samples["generated"].values
df_out.loc[:k-1, "target_preview"] = df_samples["target"].values

column_order = ["input_text", "target_summary", "target_word_count", "summary_length_generated", "target_preview", "generated"]
df_out = df_out[[col for col in column_order if col in df_out.columns]]

out_path = Path(orig_path).with_name("summarization_data_with_generated.csv")
df_out.to_csv(out_path, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

print("Guardado en:", out_path)

Guardado en: C:\Users\Alina Tatjana\OneDrive\Documentos\UVG\8vo SEMESTRE\Deep Learning\Proyecto\Proyecto\1_Data\processed\summarization_data_with_generated.csv
